# Permission 权限控制教程 (v0.6)

本教程帮助你理解 Agent 权限控制的概念和使用方式。

## 学习目标

1. 理解权限模型：Permission（权限规则）和 Policy（策略）
2. 掌握通配符匹配和 DENY 优先原则
3. 学会基于角色的访问控制（RBAC）
4. 理解如何为工具和资源添加权限保护

In [ ]:
# Step 1: 安装依赖
import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", 
    "openai", "python-dotenv", "zhipuai", "-q"
])
print("依赖安装完成！")

In [ ]:
# Step 2: 设置项目路径
import sys
from pathlib import Path

cwd = Path.cwd()
if cwd.name == "notebooks":
    project_root = cwd.parent
else:
    project_root = cwd
    while not (project_root / "pyproject.toml").exists():
        project_root = project_root.parent

src_path = project_root / "src"
sys.path.insert(0, str(src_path))

print(f"项目根目录: {project_root}")
print(f"源码路径: {src_path}")

## 1. 什么是权限控制？

权限控制决定 **谁(who)** 可以对 **什么资源(what)** 做 **什么操作(how)**。

### 为什么需要权限控制？

1. **安全边界**：Agent 不能随意执行危险操作
2. **用户授权**：敏感操作需要用户确认或预授权
3. **最小权限**：Agent 只拥有完成任务所需的最小权限

### 核心对象

| 对象 | 说明 |
|------|------|
| `Permission` | 单条权限规则 |
| `Policy` | 策略：一组权限规则，绑定到角色 |
| `AccessRequest` | 访问请求 |
| `AccessResult` | 授权结果 |

In [ ]:
# 导入 Permission 相关类型
from rogue_rabbit.contracts.permission import (
    AccessRequest,
    AccessResult,
    Effect,
    Permission,
    Policy,
)
from rogue_rabbit.core import Authorizer
from rogue_rabbit.runtime import InMemoryPolicyStore

print("导入成功！")

## 2. 创建权限规则

Permission 由三部分组成：
- **action**：操作类型（read, write, execute, delete）
- **resource**：资源标识（tool:xxx, file:xxx, memory:xxx）
- **effect**：效果（ALLOW 或 DENY）

In [ ]:
# 创建权限规则
allow_read = Permission(action="read", resource="*", effect=Effect.ALLOW)
allow_tools = Permission(action="execute", resource="tool:*", effect=Effect.ALLOW)
deny_delete = Permission(action="delete", resource="*", effect=Effect.DENY)

print("权限规则:")
for p in [allow_read, allow_tools, deny_delete]:
    print(f"  {p.effect.value:5s} | {p.action:8s} | {p.resource}")

In [ ]:
# 创建策略并绑定到角色
user_policy = Policy(
    name="user-basic",
    role="user",
    description="普通用户基本权限",
    permissions=[allow_read, allow_tools, deny_delete],
)

# 创建授权管理器
store = InMemoryPolicyStore()
authorizer = Authorizer(store=store)
authorizer.add_policy(user_policy)

print(f"策略 '{user_policy.name}' 已添加 (role={user_policy.role})")

## 3. 执行权限检查

使用 `AccessRequest` 构造请求，`authorizer.check()` 返回 `AccessResult`。

In [ ]:
# 测试各种请求
test_cases = [
    ("user", "read", "file:///data/report.txt"),
    ("user", "execute", "tool:calculator"),
    ("user", "delete", "file:///data/report.txt"),
    ("user", "write", "file:///data/report.txt"),
]

print(f"{'结果':4s} | {'角色':5s} | {'操作':8s} | {'资源'}")
print("-" * 60)
for role, action, resource in test_cases:
    request = AccessRequest(action=action, resource=resource, role=role)
    result = authorizer.check(request)
    status = "允许" if result.allowed else "拒绝"
    print(f"{status:4s} | {role:5s} | {action:8s} | {resource}")
    if not result.allowed:
        print(f"     原因: {result.reason}")

## 4. DENY 优先原则

当同时匹配到 ALLOW 和 DENY 规则时，**DENY 总是优先**。这是安全设计的基本原则。

In [ ]:
# 同时有 ALLOW *:* 和 DENY delete file:*
editor_policy = Policy(
    name="editor",
    role="editor",
    permissions=[
        Permission(action="*", resource="*", effect=Effect.ALLOW),
        Permission(action="delete", resource="file:*", effect=Effect.DENY),
    ],
)
authorizer.add_policy(editor_policy)

# 即使有 ALLOW *:*，delete 仍然被拒绝
test_cases = [
    ("editor", "read", "file:data"),
    ("editor", "write", "file:data"),
    ("editor", "delete", "file:data"),
]

print("DENY 优先原则:")
for role, action, resource in test_cases:
    request = AccessRequest(action=action, resource=resource, role=role)
    result = authorizer.check(request)
    status = "允许" if result.allowed else "拒绝"
    print(f"  {status:4s} | {action:8s} | {resource}")

## 5. 通配符匹配

资源标识支持通配符：
- `*` 匹配所有资源
- `tool:*` 匹配所有工具
- `file:///public/*` 匹配 public 目录下所有文件
- `tool:calculator` 精确匹配

In [ ]:
# 通配符匹配示例
patterns = [
    ("*", "file:///data/report.txt"),
    ("tool:*", "tool:calculator"),
    ("tool:*", "tool:file_reader"),
    ("file:///public/*", "file:///public/readme.txt"),
    ("file:///public/*", "file:///secret/keys.pem"),
    ("tool:calculator", "tool:calculator"),
    ("tool:calculator", "tool:file_reader"),
]

print(f"{'模式':25s} | {'资源':30s} | 匹配")
print("-" * 70)
for pattern, resource in patterns:
    perm = Permission(action="read", resource=pattern, effect=Effect.ALLOW)
    matched = perm.matches("read", resource)
    print(f"{pattern:25s} | {resource:30s} | {'Yes' if matched else 'No'}")

## 6. 基于角色的访问控制（RBAC）

通过策略优先级（priority）实现角色分层：guest < user < admin。

In [ ]:
# 创建 RBAC 策略
rbac_store = InMemoryPolicyStore()
rbac_authorizer = Authorizer(store=rbac_store)

# guest: 只能读公共文件和使用计算器
rbac_authorizer.add_policy(Policy(
    name="rbac-guest",
    role="guest",
    priority=1,
    permissions=[
        Permission(action="read", resource="file:///public/*", effect=Effect.ALLOW),
        Permission(action="execute", resource="tool:calculator", effect=Effect.ALLOW),
    ],
))

# user: 可读写，可执行工具，但不能删除
rbac_authorizer.add_policy(Policy(
    name="rbac-user",
    role="user",
    priority=5,
    permissions=[
        Permission(action="read", resource="*", effect=Effect.ALLOW),
        Permission(action="write", resource="memory:*", effect=Effect.ALLOW),
        Permission(action="execute", resource="tool:*", effect=Effect.ALLOW),
        Permission(action="delete", resource="*", effect=Effect.DENY),
    ],
))

# admin: 完全控制
rbac_authorizer.add_policy(Policy(
    name="rbac-admin",
    role="admin",
    priority=10,
    permissions=[
        Permission(action="*", resource="*", effect=Effect.ALLOW),
    ],
))

print("RBAC 策略已创建:")
print("  guest  (priority=1):  读公共文件 + 计算器")
print("  user   (priority=5):  读写 + 工具, 不能删除")
print("  admin  (priority=10): 完全控制")

In [ ]:
# 对比不同角色的权限
operations = [
    ("read", "file:///public/readme.txt"),
    ("execute", "tool:calculator"),
    ("write", "memory:user1"),
    ("execute", "tool:file_writer"),
    ("delete", "file:///public/old.txt"),
]

print(f"{'操作':8s} {'资源':32s} {'guest':6s} {'user':6s} {'admin':6s}")
print("-" * 70)

for action, resource in operations:
    results = []
    for role in ["guest", "user", "admin"]:
        request = AccessRequest(action=action, resource=resource, role=role)
        result = rbac_authorizer.check(request)
        results.append("允许" if result.allowed else "拒绝")
    print(f"{action:8s} {resource:32s} {results[0]:6s} {results[1]:6s} {results[2]:6s}")

## 7. 工具调用权限

在实际 Agent 场景中，权限检查嵌入在工具调用流程中。

In [ ]:
# 权限感知的工具执行器
class PermissionAwareExecutor:
    """在执行工具前检查权限"""
    
    def __init__(self, authorizer: Authorizer):
        self._authorizer = authorizer
        self._tools: dict[str, callable] = {}
    
    def register(self, name: str, func: callable):
        self._tools[name] = func
    
    def execute(self, tool_name: str, role: str, **kwargs) -> tuple[bool, str]:
        # 权限检查
        request = AccessRequest(
            action="execute",
            resource=f"tool:{tool_name}",
            role=role,
        )
        result = self._authorizer.check(request)
        
        if not result.allowed:
            return False, f"权限拒绝: {result.reason}"
        
        # 执行工具
        if tool_name not in self._tools:
            return False, f"工具不存在: {tool_name}"
        return True, self._tools[tool_name](**kwargs)

# 模拟工具
def calculator(expression: str) -> str:
    return f"计算结果: {expression} = {eval(expression)}"

def file_reader(path: str) -> str:
    return f"[读取] {path}: 文件内容..."

def file_writer(path: str, content: str) -> str:
    return f"[写入] {path}: 已写入 {len(content)} 字符"

# 设置
executor = PermissionAwareExecutor(rbac_authorizer)
executor.register("calculator", calculator)
executor.register("file_reader", file_reader)
executor.register("file_writer", file_writer)
print("工具已注册: calculator, file_reader, file_writer")

In [ ]:
# 模拟 Agent 调用工具
print("[模拟] Agent (role=user) 执行任务:\n")

# Step 1: 读取文件
success, output = executor.execute("file_reader", "user", path="/data/data.txt")
print(f"Step 1 - file_reader: {'成功' if success else '拒绝'}")
print(f"  {output}\n")

# Step 2: 计算
success, output = executor.execute("calculator", "user", expression="2 + 3")
print(f"Step 2 - calculator: {'成功' if success else '拒绝'}")
print(f"  {output}\n")

# Step 3: 写入
success, output = executor.execute("file_writer", "user", path="/output/result.txt", content="5")
print(f"Step 3 - file_writer: {'成功' if success else '拒绝'}")
print(f"  {output}")

In [ ]:
# 对比 guest 角色
print("[模拟] Agent (role=guest) 尝试调用工具:\n")

for tool_name, kwargs in [
    ("calculator", {"expression": "10 * 2"}),
    ("file_reader", {"path": "/data/data.txt"}),
    ("file_writer", {"path": "/output/result.txt", "content": "test"}),
]:
    success, output = executor.execute(tool_name, "guest", **kwargs)
    status = "成功" if success else "拒绝"
    print(f"  {status:4s} | {tool_name}")

## 8. 策略持久化

使用 FilePolicyStore 将策略保存到文件系统。

In [ ]:
from rogue_rabbit.runtime import FilePolicyStore
import tempfile

# 使用临时目录
with tempfile.TemporaryDirectory() as tmpdir:
    store_path = Path(tmpdir)
    file_store = FilePolicyStore(store_path)
    
    # 保存策略
    policy = Policy(
        name="demo-policy",
        role="user",
        priority=5,
        description="演示策略持久化",
        permissions=[
            Permission(action="read", resource="*", effect=Effect.ALLOW),
            Permission(action="delete", resource="*", effect=Effect.DENY),
        ],
    )
    file_store.save(policy)
    print(f"策略已保存: {policy.name}")
    
    # 查看文件
    files = list(store_path.glob("*.json"))
    print(f"文件: {[f.name for f in files]}")
    
    # 重新加载
    loaded = file_store.load("demo-policy")
    print(f"重新加载: {loaded.name}, 角色={loaded.role}, 权限数={len(loaded.permissions)}")
    for p in loaded.permissions:
        print(f"  {p.effect.value:5s} {p.action:8s} {p.resource}")

## 总结

### 核心概念

1. **Permission** = action + resource + effect（操作 + 资源 + 效果）
2. **Policy** = 一组 Permission 的集合，绑定到角色
3. **Authorizer** = 授权管理器，执行权限检查

### 核心原则

| 原则 | 说明 |
|------|------|
| DENY 优先 | 同时有 ALLOW 和 DENY 时 DENY 生效 |
| 默认拒绝 | 无匹配规则时自动 DENY |
| 策略优先级 | priority 高的策略优先检查 |
| 最小权限 | 只授予完成任务所需的最小权限 |

### 下一步

- 运行 `experiments/15_permission_basic.py`
- 运行 `experiments/16_tool_permission.py`
- 运行 `experiments/17_resource_permission.py`